
## Basic Azure networking

Learner Questions

* How do I deploy a network in Azure using the CLI?

Learning Objectives

* Deploy a virtual network and subnet
* Seccure virtual network using network security groups (nsg)
* Deploy ngix container
* Clean up (delete) resouces

# Setup

First we will create a virtual network with a subnet.  We will then create a network security groups 
(ngs) for the subnet and define nsg rules to let traffic in and out (ports opened). Finally we will login to (execute commands on) the containers.  After this exercise we will clean up and delete our resources.

Setup the environment variables

In [ ]:
export RGROUP="ps-rg"
export REGION="eastus"

All Azure projects start with a resouce group (RG)

In [ ]:
az group create --name $RGROUP --location $REGION

Set virtual network and subnet environment variables

In [ ]:
export VNET="VNET" # Virtual network
export SUBNET="SN" # Public facing subnet

Create a virtual network and subnet

In [ ]:
az network vnet create \
  --resource-group $RGROUP \
  --name $VNET \
  --address-prefix 10.0.0.0/16 \
  --subnet-name $SUBNET \
  --subnet-prefix 10.0.1.0/24

Create a public IP

In [ ]:
export PUBIP="PublicIPAddress" # Public IP address

In [ ]:
az network public-ip create --resource-group $RGROUP --name $PUBIP

Create network security groups

In [ ]:
export NSGROUP="Container-Security-Group" # Network Security Group

In [ ]:
az network nsg create --resource-group $RGROUP --name $NSGROUP

 Create a rule for the NSG for TCP.

In [ ]:
az network nsg rule create \
  --resource-group $RGROUP \
  --nsg-name $NSGROUP \
  --name demo-nsg \
  --access Allow \
  --protocol Tcp \
  --direction Inbound \
  --priority 300 \
  --source-address-prefix 10.0.1.0/24 \
  --source-port-range "*" \
  --destination-address-prefix "*" \
  --destination-port-range 80 8080 443

Add rule to block all traffic from blah, Note --priority 100 is lower than the previous

In [ ]:
az network nsg rule create \
  --resource-group $RGROUP \
  --nsg-name $NSGROUP \
  --name denyAll \
  --access Deny \
  --protocol Tcp \
  --direction Inbound \
  --priority 100 \
  --source-address-prefix "*" \
  --source-port-range "*" \
  --destination-address-prefix "*" \
  --destination-port-range "*"

Associate the NSG to the subnet.

In [ ]:
az network vnet subnet update \
  --vnet-name $VNET \
  --name $SUBNET \
  --resource-group $RGROUP \
  --network-security-group $NSGROUP

List NSG

In [ ]:
az network nsg list

**Deploy container**

Create container

In [ ]:
export CONT="demo-container" # Public perfSONAR container

In [ ]:
az container create \
  --resource-group $RGROUP \
  --name $CONT \
  --image nginx \
  --restart-policy never \
  --vnet $VNET \
  --subnet $SUBNET

Get front container IP address

In [ ]:
az container show --resource-group $RGROUP   --name $CONT --query ipAddress.ip --output tsv

List containers

In [ ]:
az container list --resource-group $RGROUP --output table

Get logs from backend perfSONAR container

In [ ]:
az container logs --resource-group $RGROUP --name $CONT

Stream logs from backend perfSONAR container

In [ ]:
az container attach --resource-group $RGROUP --name $SCONT

Execute command on contaioner

"Login" into container with bash shell

In [ ]:
az container exec --resource-group $RGROUP --name $CONT --exec-command "/bin/bash"

Now execute "ls" command to see what is there.

Delete virtual network

In [ ]:
az network vnet delete --resource-group $RGROUP --name $VNET

Clean up: delete container & rg

In [ ]:
az container delete --resource-group $RGROUP --name <containers> -y

az group delete --name $RGROUP